# GPU Parallel Computing (CUDA Python) — 50 Tasks
### Zeba Academy — numba.cuda & CuPy, Python 3.10+

Run **Cell 1** first in Google Colab with a T4/L4/A100 GPU runtime
(`Runtime -> Change runtime type -> GPU`).

Each module below is self-contained: kernel + host wrapper + inline
correctness check against a NumPy/SciPy reference, matching the
task's `Launch & Verification` contract from the tasksheet.


In [ ]:
# Cell 1: Environment Provisioning & GPU Verification
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv
!pip install -q numba cupy-cuda12x pytest scipy

import numba.cuda as cuda
import cupy as cp
import numpy as np
import math
import time

if not cuda.is_available():
    raise SystemError("CRITICAL: CUDA GPU not detected! Runtime > Change runtime type > GPU.")

gpu = cuda.get_current_device()
print(f"\n[+] Environment Certified for Autonomous CUDA Lab")
print(f"  Target Device    : {gpu.name.decode('utf-8')}")
print(f"  Compute Capable  : {gpu.compute_capability}")
print(f"  Warp Size        : {gpu.WARP_SIZE} threads")
print(f"  Max Block Threads: {gpu.MAX_THREADS_PER_BLOCK}")
print(f"  Max Shared/Block : {gpu.MAX_SHARED_MEMORY_PER_BLOCK / 1024:.1f} KB\n")


## Module 1: CUDA Execution Model (Tasks 1–8)

In [ ]:
"""
Module 1: CUDA Execution Model
Tasks 1-8: Thread indexing, 1D/2D/3D grids

Zeba Academy GPU Parallel Computing Lab
"""

import math
import numpy as np
from numba import cuda


# =============================================================================
# Task 01: 1D Vector Addition with Boundary Checks
# Signature: vector_add_1d(a, b, out)
# C[i] = A[i] + B[i]
# =============================================================================
@cuda.jit
def _vector_add_1d_kernel(a, b, out):
    i = cuda.grid(1)
    if i < a.shape[0]:
        out[i] = a[i] + b[i]


def vector_add_1d(a, b, out):
    """Host wrapper: element-wise addition of two 1D float32 arrays."""
    n = a.shape[0]
    threads_per_block = 256
    blocks_per_grid = (n + threads_per_block - 1) // threads_per_block

    d_a = cuda.to_device(a)
    d_b = cuda.to_device(b)
    d_out = cuda.device_array_like(out)

    _vector_add_1d_kernel[blocks_per_grid, threads_per_block](d_a, d_b, d_out)
    cuda.synchronize()

    d_out.copy_to_host(out)
    return out


# =============================================================================
# Task 02: 1D Grid-Stride Loop Element-wise Scaling
# Signature: vector_scale_grid_stride(x, alpha, out)
# Launch with FIXED grid dims decoupled from N; loop with stride.
# =============================================================================
@cuda.jit
def _vector_scale_grid_stride_kernel(x, alpha, out):
    start = cuda.grid(1)
    stride = cuda.gridsize(1)
    n = x.shape[0]
    for i in range(start, n, stride):
        out[i] = x[i] * alpha


def vector_scale_grid_stride(x, alpha, out):
    """Host wrapper: scales x by alpha using a grid-stride loop.

    Grid size is deliberately fixed and smaller than N (decoupled from
    arbitrary input length), per the task's hardware constraint.
    """
    threads_per_block = 256
    blocks_per_grid = 64  # fixed, intentionally << N for large N

    d_x = cuda.to_device(x)
    d_out = cuda.device_array_like(out)

    _vector_scale_grid_stride_kernel[blocks_per_grid, threads_per_block](
        d_x, alpha, d_out
    )
    cuda.synchronize()

    d_out.copy_to_host(out)
    return out


# =============================================================================
# Task 03: 2D Matrix Element-wise Addition & Index Mapping
# Signature: matrix_add_2d(A, B, C)
# row = blockIdx.y * blockDim.y + threadIdx.y
# col = blockIdx.x * blockDim.x + threadIdx.x
# =============================================================================
@cuda.jit
def _matrix_add_2d_kernel(A, B, C):
    col = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
    row = cuda.blockIdx.y * cuda.blockDim.y + cuda.threadIdx.y
    M, N = A.shape
    if row < M and col < N:
        C[row, col] = A[row, col] + B[row, col]


def matrix_add_2d(A, B, C):
    """Host wrapper: element-wise addition of 2D matrices (M, N)."""
    M, N = A.shape
    threads_2d = (16, 16)
    blocks_2d = (
        (N + threads_2d[0] - 1) // threads_2d[0],
        (M + threads_2d[1] - 1) // threads_2d[1],
    )

    d_A = cuda.to_device(A)
    d_B = cuda.to_device(B)
    d_C = cuda.device_array_like(C)

    _matrix_add_2d_kernel[blocks_2d, threads_2d](d_A, d_B, d_C)
    cuda.synchronize()

    d_C.copy_to_host(C)
    return C


# =============================================================================
# Task 04: 2D Matrix Transpose (Naive Global Memory)
# Signature: matrix_transpose_naive(A, B)
# B[col, row] = A[row, col]; A is (M, N), B is (N, M)
# =============================================================================
@cuda.jit
def _matrix_transpose_naive_kernel(A, B):
    col = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
    row = cuda.blockIdx.y * cuda.blockDim.y + cuda.threadIdx.y
    M, N = A.shape
    if row < M and col < N:
        B[col, row] = A[row, col]


def matrix_transpose_naive(A, B):
    """Host wrapper: transposes A (M, N) into B (N, M) via naive global memory."""
    M, N = A.shape
    threads_2d = (16, 16)
    blocks_2d = (
        (N + threads_2d[0] - 1) // threads_2d[0],
        (M + threads_2d[1] - 1) // threads_2d[1],
    )

    d_A = cuda.to_device(A)
    d_B = cuda.device_array_like(B)

    _matrix_transpose_naive_kernel[blocks_2d, threads_2d](d_A, d_B)
    cuda.synchronize()

    d_B.copy_to_host(B)
    return B


# =============================================================================
# Task 05: 3D Grid Volume Initialization & Index Flattening
# Signature: volume_init_3d(vol, D, H, W)
# vol[z, y, x] = z * 10000 + y * 100 + x
# =============================================================================
@cuda.jit
def _volume_init_3d_kernel(vol, D, H, W):
    x, y, z = cuda.grid(3)
    if x < W and y < H and z < D:
        vol[z, y, x] = z * 10000 + y * 100 + x


def volume_init_3d(vol, D, H, W):
    """Host wrapper: initializes a (D, H, W) volume with flattened index values."""
    threads_3d = (8, 8, 8)
    blocks_3d = (
        (W + threads_3d[0] - 1) // threads_3d[0],
        (H + threads_3d[1] - 1) // threads_3d[1],
        (D + threads_3d[2] - 1) // threads_3d[2],
    )

    d_vol = cuda.to_device(vol)

    _volume_init_3d_kernel[blocks_3d, threads_3d](d_vol, D, H, W)
    cuda.synchronize()

    d_vol.copy_to_host(vol)
    return vol


# =============================================================================
# Task 06: Color Image to Grayscale Luminance Conversion
# Signature: rgb_to_grayscale(img_rgb, img_gray)
# Y = 0.299*R + 0.587*G + 0.114*B  (ITU-R BT.601)
# =============================================================================
@cuda.jit
def _rgb_to_grayscale_kernel(img_rgb, img_gray):
    x, y = cuda.grid(2)
    H, W = img_gray.shape
    if y < H and x < W:
        r = img_rgb[y, x, 0]
        g = img_rgb[y, x, 1]
        b = img_rgb[y, x, 2]
        img_gray[y, x] = 0.299 * r + 0.587 * g + 0.114 * b


def rgb_to_grayscale(img_rgb, img_gray):
    """Host wrapper: converts (H, W, 3) RGB image to (H, W) grayscale."""
    H, W, _ = img_rgb.shape
    threads_2d = (16, 16)
    blocks_2d = (
        (W + threads_2d[0] - 1) // threads_2d[0],
        (H + threads_2d[1] - 1) // threads_2d[1],
    )

    d_rgb = cuda.to_device(img_rgb)
    d_gray = cuda.device_array_like(img_gray)

    _rgb_to_grayscale_kernel[blocks_2d, threads_2d](d_rgb, d_gray)
    cuda.synchronize()

    d_gray.copy_to_host(img_gray)
    return img_gray


# =============================================================================
# Task 07: Vector Linear Interpolation (LERP) with Dynamic Step Size
# Signature: vector_lerp(a, b, t, out)
# out[i] = (1 - t[i]) * a[i] + t[i] * b[i]
# =============================================================================
@cuda.jit
def _vector_lerp_kernel(a, b, t, out):
    i = cuda.grid(1)
    if i < a.shape[0]:
        out[i] = (1.0 - t[i]) * a[i] + t[i] * b[i]


def vector_lerp(a, b, t, out):
    """Host wrapper: element-wise LERP between a and b using per-element weight t."""
    n = a.shape[0]
    threads_per_block = 256
    blocks_per_grid = (n + threads_per_block - 1) // threads_per_block

    d_a = cuda.to_device(a)
    d_b = cuda.to_device(b)
    d_t = cuda.to_device(t)
    d_out = cuda.device_array_like(out)

    _vector_lerp_kernel[blocks_per_grid, threads_per_block](d_a, d_b, d_t, d_out)
    cuda.synchronize()

    d_out.copy_to_host(out)
    return out


# =============================================================================
# Task 08: Thresholding & Boolean Mask Generation
# Signature: vector_threshold(x, thresh, mask)
# mask[i] = 1 if x[i] >= thresh else 0  (branchless predication via arithmetic)
# =============================================================================
@cuda.jit
def _vector_threshold_kernel(x, thresh, mask):
    start = cuda.grid(1)
    stride = cuda.gridsize(1)
    n = x.shape[0]
    for i in range(start, n, stride):
        # Predicated assignment avoids warp-divergent if/else branches:
        # the boolean->int cast compiles to a select/setp instruction
        # rather than a branch.
        mask[i] = np.uint8(1) if x[i] >= thresh else np.uint8(0)


def vector_threshold(x, thresh, mask):
    """Host wrapper: generates a uint8 boolean mask via grid-stride loop."""
    n = x.shape[0]
    threads_per_block = 256
    blocks_per_grid = 64  # grid-stride, decoupled from N

    d_x = cuda.to_device(x)
    d_mask = cuda.device_array_like(mask)

    _vector_threshold_kernel[blocks_per_grid, threads_per_block](d_x, thresh, d_mask)
    cuda.synchronize()

    d_mask.copy_to_host(mask)
    return mask


# =============================================================================


In [ ]:
# --- Module 1 Verification ---
print("Module 1: CUDA Execution Model — verification")

for n in [1, 255, 256, 1024, 1_000_003]:
    a = np.random.rand(n).astype(np.float32)
    b = np.random.rand(n).astype(np.float32)
    out = np.empty_like(a)
    vector_add_1d(a, b, out)
    np.testing.assert_allclose(out, a + b, atol=1e-5)
print("[PASS] Task 1: vector_add_1d")

n = 10_000_000
x = np.random.rand(n).astype(np.float32)
alpha = 2.5
out = np.empty_like(x)
vector_scale_grid_stride(x, alpha, out)
np.testing.assert_allclose(out, x * alpha, atol=1e-5)
print("[PASS] Task 2: vector_scale_grid_stride")

M, N = 513, 1021
A = np.random.rand(M, N).astype(np.float32)
B = np.random.rand(M, N).astype(np.float32)
C = np.empty_like(A)
matrix_add_2d(A, B, C)
np.testing.assert_allclose(C, A + B, atol=1e-5)
print("[PASS] Task 3: matrix_add_2d")

A = np.random.rand(M, N).astype(np.float32)
Bt = np.empty((N, M), dtype=np.float32)
matrix_transpose_naive(A, Bt)
np.testing.assert_allclose(Bt, A.T, atol=1e-5)
print("[PASS] Task 4: matrix_transpose_naive")

D, H, W = 33, 47, 65
vol = np.zeros((D, H, W), dtype=np.int64)
volume_init_3d(vol, D, H, W)
zz, yy, xx = np.meshgrid(np.arange(D), np.arange(H), np.arange(W), indexing="ij")
np.testing.assert_equal(vol, zz * 10000 + yy * 100 + xx)
print("[PASS] Task 5: volume_init_3d")

H, W = 128, 128
img_rgb = (np.random.rand(H, W, 3) * 255).astype(np.float32)
img_gray = np.empty((H, W), dtype=np.float32)
rgb_to_grayscale(img_rgb, img_gray)
expected = 0.299 * img_rgb[:, :, 0] + 0.587 * img_rgb[:, :, 1] + 0.114 * img_rgb[:, :, 2]
np.testing.assert_allclose(img_gray, expected, atol=1e-4)
print("[PASS] Task 6: rgb_to_grayscale")

n = 10_000
a = np.random.rand(n).astype(np.float32)
b = np.random.rand(n).astype(np.float32)
t = np.random.rand(n).astype(np.float32)
out = np.empty_like(a)
vector_lerp(a, b, t, out)
np.testing.assert_allclose(out, (1 - t) * a + t * b, atol=1e-5)
print("[PASS] Task 7: vector_lerp")

x = np.random.rand(n).astype(np.float32)
thresh = 0.5
mask = np.empty(n, dtype=np.uint8)
vector_threshold(x, thresh, mask)
np.testing.assert_equal(mask, (x >= thresh).astype(np.uint8))
print("[PASS] Task 8: vector_threshold")

print("\nModule 1: all 8 tasks passed.")


## Module 2: Memory Hierarchy & Shared Memory (Tasks 9–17)

In [ ]:
# =============================================================================
# Module 2: Memory Hierarchy & Shared Memory (Tasks 9-17)
# =============================================================================
from numba import cuda, float32
import numpy as np
import time

TPB = 256  # threads per block, used across module 2


# --- Task 09: Coalesced vs Strided Memory Access Benchmark ---
@cuda.jit
def _copy_strided_kernel(in_arr, out_arr, stride):
    i = cuda.grid(1)
    src = (i * stride) % in_arr.shape[0]
    if i < out_arr.shape[0]:
        out_arr[i] = in_arr[src]


def benchmark_strided_access(in_arr, out_arr, stride):
    """Copies in_arr -> out_arr reading with the given stride; returns kernel time (ms)."""
    n = out_arr.shape[0]
    bpg = (n + TPB - 1) // TPB
    d_in = cuda.to_device(in_arr)
    d_out = cuda.device_array_like(out_arr)

    _copy_strided_kernel[bpg, TPB](d_in, d_out, stride)
    cuda.synchronize()

    start = cuda.event()
    end = cuda.event()
    start.record()
    _copy_strided_kernel[bpg, TPB](d_in, d_out, stride)
    end.record()
    end.synchronize()
    elapsed_ms = cuda.event_elapsed_time(start, end)

    d_out.copy_to_host(out_arr)
    return elapsed_ms


# --- Task 10: 1D Stencil / Moving Average with Shared Memory Halo ---
_stencil_kernel_cache = {}


def _make_stencil_kernel(radius, block_size=TPB):
    @cuda.jit
    def kernel(in_arr, out_arr):
        shared = cuda.shared.array(shape=(block_size + 2 * radius,), dtype=float32)
        tid = cuda.threadIdx.x
        gid = cuda.grid(1)
        n = in_arr.shape[0]

        # center element (clamped so tail threads of last block still fill the tile)
        shared[tid + radius] = in_arr[min(gid, n - 1)]
        # left halo
        if tid < radius:
            src = gid - radius
            src = 0 if src < 0 else src
            shared[tid] = in_arr[min(src, n - 1)]
        # right halo
        if tid >= block_size - radius:
            src = gid + radius
            src = n - 1 if src >= n else src
            shared[tid + 2 * radius] = in_arr[max(src, 0)]
        cuda.syncthreads()

        if gid < n:
            acc = 0.0
            for k in range(-radius, radius + 1):
                acc += shared[tid + radius + k]
            out_arr[gid] = acc / (2 * radius + 1)

    return kernel


def stencil_1d_shared(in_arr, out_arr, radius):
    """Host wrapper: moving-average filter with shared-memory halo tiling."""
    kernel = _stencil_kernel_cache.setdefault(radius, _make_stencil_kernel(radius))
    n = in_arr.shape[0]
    bpg = (n + TPB - 1) // TPB

    d_in = cuda.to_device(in_arr.astype(np.float32))
    d_out = cuda.device_array_like(out_arr.astype(np.float32))
    kernel[bpg, TPB](d_in, d_out)
    cuda.synchronize()
    d_out.copy_to_host(out_arr)
    return out_arr


# --- Task 11: 1D Convolution with Static Shared Memory Tiling ---
_conv_kernel_cache = {}


def _make_conv_kernel(ksize, block_size=TPB):
    radius = ksize // 2

    @cuda.jit
    def kernel(x, kern, y):
        shared = cuda.shared.array(shape=(block_size + 2 * radius,), dtype=float32)
        tid = cuda.threadIdx.x
        gid = cuda.grid(1)
        n = x.shape[0]

        shared[tid + radius] = x[gid] if gid < n else 0.0
        if tid < radius:
            src = gid - radius
            shared[tid] = x[src] if 0 <= src < n else 0.0
        if tid >= block_size - radius:
            src = gid + radius
            shared[tid + 2 * radius] = x[src] if 0 <= src < n else 0.0
        cuda.syncthreads()

        if gid < n:
            acc = 0.0
            for k in range(ksize):
                acc += shared[tid + k] * kern[ksize - 1 - k]
            y[gid] = acc

    return kernel


def conv_1d_shared_tiled(x, kernel, y):
    """Host wrapper: 'same'-mode 1D convolution using shared-memory tiling."""
    ksize = kernel.shape[0]
    cfn = _conv_kernel_cache.setdefault(ksize, _make_conv_kernel(ksize))
    n = x.shape[0]
    bpg = (n + TPB - 1) // TPB

    d_x = cuda.to_device(x.astype(np.float32))
    d_k = cuda.to_device(kernel.astype(np.float32))
    d_y = cuda.device_array_like(y.astype(np.float32))
    cfn[bpg, TPB](d_x, d_k, d_y)
    cuda.synchronize()
    d_y.copy_to_host(y)
    return y


# --- Task 12: 2D Tiled Matrix Multiplication ---
TILE_DIM = 16


@cuda.jit
def _matmul_tiled_kernel(A, B, C):
    sA = cuda.shared.array((TILE_DIM, TILE_DIM), dtype=float32)
    sB = cuda.shared.array((TILE_DIM, TILE_DIM), dtype=float32)

    row = cuda.blockIdx.y * TILE_DIM + cuda.threadIdx.y
    col = cuda.blockIdx.x * TILE_DIM + cuda.threadIdx.x
    M, K = A.shape
    _, N = B.shape

    acc = 0.0
    n_tiles = (K + TILE_DIM - 1) // TILE_DIM
    for t in range(n_tiles):
        a_col = t * TILE_DIM + cuda.threadIdx.x
        b_row = t * TILE_DIM + cuda.threadIdx.y

        sA[cuda.threadIdx.y, cuda.threadIdx.x] = (
            A[row, a_col] if row < M and a_col < K else 0.0
        )
        sB[cuda.threadIdx.y, cuda.threadIdx.x] = (
            B[b_row, col] if b_row < K and col < N else 0.0
        )
        cuda.syncthreads()

        for k in range(TILE_DIM):
            acc += sA[cuda.threadIdx.y, k] * sB[k, cuda.threadIdx.x]
        cuda.syncthreads()

    if row < M and col < N:
        C[row, col] = acc


def matmul_tiled_shared(A, B, C):
    """Host wrapper: C = A @ B using TILE_DIM x TILE_DIM shared-memory tiles."""
    M, K = A.shape
    _, N = B.shape
    blocks = ((N + TILE_DIM - 1) // TILE_DIM, (M + TILE_DIM - 1) // TILE_DIM)

    d_A = cuda.to_device(A.astype(np.float32))
    d_B = cuda.to_device(B.astype(np.float32))
    d_C = cuda.device_array_like(C.astype(np.float32))
    _matmul_tiled_kernel[blocks, (TILE_DIM, TILE_DIM)](d_A, d_B, d_C)
    cuda.synchronize()
    d_C.copy_to_host(C)
    return C


# --- Task 13: 2D Bank Conflict-Free Matrix Transpose ---
TDIM = 32


@cuda.jit
def _transpose_bcf_kernel(A, B):
    tile = cuda.shared.array((TDIM, TDIM + 1), dtype=float32)  # +1 padding

    x = cuda.blockIdx.x * TDIM + cuda.threadIdx.x
    y = cuda.blockIdx.y * TDIM + cuda.threadIdx.y
    M, N = A.shape

    if y < M and x < N:
        tile[cuda.threadIdx.y, cuda.threadIdx.x] = A[y, x]
    cuda.syncthreads()

    # transposed output coordinates
    out_x = cuda.blockIdx.y * TDIM + cuda.threadIdx.x
    out_y = cuda.blockIdx.x * TDIM + cuda.threadIdx.y
    if out_y < N and out_x < M:
        B[out_y, out_x] = tile[cuda.threadIdx.x, cuda.threadIdx.y]


def transpose_bank_conflict_free(A, B):
    """Host wrapper: transposes A using a +1-padded 32x32 shared tile."""
    M, N = A.shape
    blocks = ((N + TDIM - 1) // TDIM, (M + TDIM - 1) // TDIM)

    d_A = cuda.to_device(A.astype(np.float32))
    d_B = cuda.device_array_like(B.astype(np.float32))
    _transpose_bcf_kernel[blocks, (TDIM, TDIM)](d_A, d_B)
    cuda.synchronize()
    d_B.copy_to_host(B)
    return B


# --- Task 14: Parallel Tree Sum Reduction in Shared Memory ---
@cuda.jit
def _reduction_sum_kernel(in_arr, block_sums):
    shared = cuda.shared.array(shape=(TPB,), dtype=float32)
    tid = cuda.threadIdx.x
    gid = cuda.grid(1)
    n = in_arr.shape[0]

    shared[tid] = in_arr[gid] if gid < n else 0.0
    cuda.syncthreads()

    stride = TPB // 2
    while stride > 0:
        if tid < stride:
            shared[tid] += shared[tid + stride]
        cuda.syncthreads()
        stride >>= 1

    if tid == 0:
        block_sums[cuda.blockIdx.x] = shared[0]


def reduction_sum_shared(in_arr, block_sums):
    """Host wrapper: block-level tree-sum reduction, one partial sum per block."""
    n = in_arr.shape[0]
    bpg = (n + TPB - 1) // TPB

    d_in = cuda.to_device(in_arr.astype(np.float32))
    d_sums = cuda.device_array_like(block_sums.astype(np.float32))
    _reduction_sum_kernel[bpg, TPB](d_in, d_sums)
    cuda.synchronize()
    d_sums.copy_to_host(block_sums)
    return block_sums


# --- Task 15: Parallel Min/Max (Argmax) Reduction with Index Tracking ---
@cuda.jit
def _reduction_argmax_kernel(in_arr, block_max_vals, block_max_indices):
    s_val = cuda.shared.array(shape=(TPB,), dtype=float32)
    s_idx = cuda.shared.array(shape=(TPB,), dtype=np.int64)
    tid = cuda.threadIdx.x
    gid = cuda.grid(1)
    n = in_arr.shape[0]

    if gid < n:
        s_val[tid] = in_arr[gid]
        s_idx[tid] = gid
    else:
        s_val[tid] = -3.4e38
        s_idx[tid] = -1
    cuda.syncthreads()

    stride = TPB // 2
    while stride > 0:
        if tid < stride:
            ov = s_val[tid + stride]
            oi = s_idx[tid + stride]
            if ov > s_val[tid] or (ov == s_val[tid] and oi >= 0 and oi < s_idx[tid]):
                s_val[tid] = ov
                s_idx[tid] = oi
        cuda.syncthreads()
        stride >>= 1

    if tid == 0:
        block_max_vals[cuda.blockIdx.x] = s_val[0]
        block_max_indices[cuda.blockIdx.x] = s_idx[0]


def reduction_argmax_shared(in_arr, block_max_vals, block_max_indices):
    """Host wrapper: per-block max value + its global index."""
    n = in_arr.shape[0]
    bpg = (n + TPB - 1) // TPB

    d_in = cuda.to_device(in_arr.astype(np.float32))
    d_vals = cuda.device_array_like(block_max_vals.astype(np.float32))
    d_idx = cuda.device_array_like(block_max_indices.astype(np.int64))
    _reduction_argmax_kernel[bpg, TPB](d_in, d_vals, d_idx)
    cuda.synchronize()
    d_vals.copy_to_host(block_max_vals)
    d_idx.copy_to_host(block_max_indices)
    return block_max_vals, block_max_indices


# --- Task 16: Constant Memory Broadcast Filter (64-tap FIR) ---
FIR_WEIGHTS = (np.hanning(64) / np.hanning(64).sum()).astype(np.float32)  # default 64-tap filter
_const_filter_cache = {}


def _make_constant_filter_kernel(weights):
    K = weights.shape[0]
    radius = K // 2
    c_weights = weights.astype(np.float32)

    @cuda.jit
    def kernel(signal, out):
        w = cuda.const.array_like(c_weights)
        i = cuda.grid(1)
        n = signal.shape[0]
        if i < n:
            acc = 0.0
            for k in range(K):
                src = i + k - radius
                if 0 <= src < n:
                    acc += signal[src] * w[K - 1 - k]
            out[i] = acc

    return kernel


def constant_memory_filter(signal, out, weights=None):
    """Host wrapper: 64-tap FIR filter with weights broadcast from constant memory."""
    if weights is None:
        weights = FIR_WEIGHTS
    key = weights.tobytes()
    kernel = _const_filter_cache.get(key)
    if kernel is None:
        kernel = _make_constant_filter_kernel(weights)
        _const_filter_cache[key] = kernel

    n = signal.shape[0]
    bpg = (n + TPB - 1) // TPB
    d_sig = cuda.to_device(signal.astype(np.float32))
    d_out = cuda.device_array_like(out.astype(np.float32))
    kernel[bpg, TPB](d_sig, d_out)
    cuda.synchronize()
    d_out.copy_to_host(out)
    return out


# --- Task 17: Pinned vs Pageable Host Memory Transfer ---
def benchmark_pinned_transfer(size_bytes):
    """Benchmarks H2D+D2H bandwidth for pageable vs pinned host buffers.

    Returns (bw_pageable_GBs, bw_pinned_GBs).
    """
    n_floats = size_bytes // 4
    pageable = np.random.rand(n_floats).astype(np.float32)
    pinned = cuda.pinned_array(n_floats, dtype=np.float32)
    pinned[:] = pageable

    def _roundtrip(host_arr, trials=5):
        d = cuda.to_device(host_arr)  # warmup alloc
        cuda.synchronize()
        start = time.perf_counter()
        for _ in range(trials):
            d = cuda.to_device(host_arr)
            out = d.copy_to_host()
        cuda.synchronize()
        elapsed = (time.perf_counter() - start) / trials
        total_bytes = host_arr.nbytes * 2  # H2D + D2H
        return (total_bytes / elapsed) / 1e9  # GB/s

    bw_pageable = _roundtrip(pageable)
    bw_pinned = _roundtrip(pinned)
    return bw_pageable, bw_pinned


In [ ]:
# --- Module 2 Verification ---
import scipy.ndimage

print("Module 2: Memory Hierarchy & Shared Memory — verification")

# Task 9
n = 1_000_000
in_arr = np.random.rand(n).astype(np.float32)
out_arr = np.empty_like(in_arr)
t1 = benchmark_strided_access(in_arr, out_arr, 1)
t32 = benchmark_strided_access(in_arr, out_arr, 32)
print(f"[PASS] Task 9: benchmark_strided_access (stride1={t1:.3f}ms, stride32={t32:.3f}ms)")

# Task 10
n, radius = 100_007, 3
in_arr = np.sin(np.linspace(0, 20, n)).astype(np.float32)
out_arr = np.empty_like(in_arr)
stencil_1d_shared(in_arr, out_arr, radius)
ref = scipy.ndimage.uniform_filter1d(in_arr, size=2 * radius + 1, mode="nearest")
np.testing.assert_allclose(out_arr, ref, atol=1e-4)
print("[PASS] Task 10: stencil_1d_shared")

# Task 11
n = 5000
x = np.random.rand(n).astype(np.float32)
kern = np.array([0.1, 0.2, 0.4, 0.2, 0.1], dtype=np.float32)
y = np.empty_like(x)
conv_1d_shared_tiled(x, kern, y)
ref = np.convolve(x, kern, mode="same")
np.testing.assert_allclose(y, ref, atol=1e-3)
print("[PASS] Task 11: conv_1d_shared_tiled")

# Task 12
for (M, K, N) in [(64, 64, 64), (130, 90, 77)]:
    A = np.random.rand(M, K).astype(np.float32)
    B = np.random.rand(K, N).astype(np.float32)
    C = np.empty((M, N), dtype=np.float32)
    matmul_tiled_shared(A, B, C)
    np.testing.assert_allclose(C, A @ B, atol=1e-2, rtol=1e-3)
print("[PASS] Task 12: matmul_tiled_shared")

# Task 13
M, N = 257, 129
A = np.random.rand(M, N).astype(np.float32)
B = np.empty((N, M), dtype=np.float32)
transpose_bank_conflict_free(A, B)
np.testing.assert_allclose(B, A.T, atol=1e-5)
print("[PASS] Task 13: transpose_bank_conflict_free")

# Task 14
n = 1_000_000
in_arr = np.random.rand(n).astype(np.float32)
n_blocks = (n + TPB - 1) // TPB
block_sums = np.empty(n_blocks, dtype=np.float32)
reduction_sum_shared(in_arr, block_sums)
np.testing.assert_allclose(np.sum(block_sums), np.sum(in_arr), rtol=1e-4)
print("[PASS] Task 14: reduction_sum_shared")

# Task 15
n = 100_000
in_arr = np.random.rand(n).astype(np.float32)
n_blocks = (n + TPB - 1) // TPB
block_max_vals = np.empty(n_blocks, dtype=np.float32)
block_max_idx = np.empty(n_blocks, dtype=np.int64)
reduction_argmax_shared(in_arr, block_max_vals, block_max_idx)
best = np.argmax(block_max_vals)
assert np.isclose(block_max_vals[best], np.max(in_arr))
assert in_arr[block_max_idx[best]] == np.max(in_arr)
print("[PASS] Task 15: reduction_argmax_shared")

# Task 16
n = 5000
weights = np.random.rand(64).astype(np.float32)
signal = np.random.rand(n).astype(np.float32)
out = np.empty_like(signal)
constant_memory_filter(signal, out, weights)
ref = np.convolve(signal, weights, mode="same")
np.testing.assert_allclose(out, ref, atol=1e-2, rtol=1e-2)
print("[PASS] Task 16: constant_memory_filter")

# Task 17
bw_pageable, bw_pinned = benchmark_pinned_transfer(16 * 1024 * 1024)
print(f"[PASS] Task 17: benchmark_pinned_transfer (pageable={bw_pageable:.2f} GB/s, pinned={bw_pinned:.2f} GB/s)")

print("\nModule 2: all 9 tasks passed.")


## Module 3: Warp Intrinsics, Synchronization & Atomics (Tasks 18–25)

In [ ]:
# =============================================================================
# Module 3: Warp Intrinsics, Synchronization & Atomics (Tasks 18-25)
# =============================================================================
from numba import cuda, int32, int64, float32
import numpy as np

WARP = 32


# --- Task 18: Warp-Level Reduction using Shuffle Intrinsics ---
@cuda.jit
def _warp_reduce_sum_kernel(in_arr, out_arr):
    i = cuda.grid(1)
    val = in_arr[i] if i < in_arr.shape[0] else 0.0

    mask = np.uint32(0xFFFFFFFF)
    for delta in (16, 8, 4, 2, 1):
        val += cuda.shfl_down_sync(mask, val, delta)

    lane = cuda.threadIdx.x % WARP
    if lane == 0:
        cuda.atomic.add(out_arr, 0, val)


def warp_reduce_sum_shfl(in_arr, out_arr):
    """Host wrapper: sums in_arr using warp shuffle-down reduction, no shared mem."""
    n = in_arr.shape[0]
    bpg = (n + TPB - 1) // TPB
    d_in = cuda.to_device(in_arr.astype(np.float32))
    d_out = cuda.to_device(np.zeros(1, dtype=np.float32))
    _warp_reduce_sum_kernel[bpg, TPB](d_in, d_out)
    cuda.synchronize()
    d_out.copy_to_host(out_arr)
    return out_arr


# --- Task 19: Warp Vote Functions & Branch Divergence Counter ---
@cuda.jit
def _warp_vote_kernel(data, mask_out):
    i = cuda.grid(1)
    mask = np.uint32(0xFFFFFFFF)
    pred = data[i] > 0.0 if i < data.shape[0] else False

    all_true = cuda.all_sync(mask, pred)
    any_true = cuda.any_sync(mask, pred)
    ballot = cuda.ballot_sync(mask, pred)

    lane = cuda.threadIdx.x % WARP
    if lane == 0:
        warp_id = i // WARP
        if warp_id < mask_out.shape[0]:
            mask_out[warp_id] = ballot


def warp_vote_analyzer(data, mask_out):
    """Host wrapper: per-warp ballot mask of predicate (data[i] > 0)."""
    n = data.shape[0]
    bpg = (n + TPB - 1) // TPB
    d_data = cuda.to_device(data.astype(np.float32))
    d_mask = cuda.device_array_like(mask_out.astype(np.uint32))
    _warp_vote_kernel[bpg, TPB](d_data, d_mask)
    cuda.synchronize()
    d_mask.copy_to_host(mask_out)
    return mask_out


# --- Task 20: 1D Inclusive Prefix Sum (Hillis-Steele Scan) ---
@cuda.jit
def _scan_hillis_steele_kernel(x, out, n):
    # double buffer: buf[0, :] and buf[1, :], ping-pong selected by `pin`
    buf = cuda.shared.array(shape=(2, 1024), dtype=float32)
    tid = cuda.threadIdx.x

    buf[0, tid] = x[tid] if tid < n else 0.0
    cuda.syncthreads()

    pin = 0
    stride = 1
    while stride < n:
        pout = 1 - pin
        if tid >= stride:
            buf[pout, tid] = buf[pin, tid] + buf[pin, tid - stride]
        else:
            buf[pout, tid] = buf[pin, tid]
        cuda.syncthreads()
        pin = pout
        stride *= 2

    if tid < n:
        out[tid] = buf[pin, tid]


def scan_hillis_steele(x, out):
    """Host wrapper: single-block inclusive scan (arrays up to 1024 elements)."""
    n = x.shape[0]
    assert n <= 1024
    d_x = cuda.to_device(x.astype(np.float32))
    d_out = cuda.device_array_like(out.astype(np.float32))
    _scan_hillis_steele_kernel[1, n](d_x, d_out, n)
    cuda.synchronize()
    d_out.copy_to_host(out)
    return out


# --- Task 21: Work-Efficient Parallel Prefix Sum (Blelloch Scan) ---
@cuda.jit
def _scan_blelloch_kernel(x, out, n, padded_n):
    shared = cuda.shared.array(shape=(2048,), dtype=float32)
    tid = cuda.threadIdx.x

    shared[2 * tid] = x[2 * tid] if 2 * tid < n else 0.0
    shared[2 * tid + 1] = x[2 * tid + 1] if 2 * tid + 1 < n else 0.0
    cuda.syncthreads()

    # Up-sweep (reduce)
    offset = 1
    d = padded_n // 2
    while d > 0:
        if tid < d:
            ai = offset * (2 * tid + 1) - 1
            bi = offset * (2 * tid + 2) - 1
            shared[bi] += shared[ai]
        cuda.syncthreads()
        offset *= 2
        d //= 2

    if tid == 0:
        shared[padded_n - 1] = 0.0
    cuda.syncthreads()

    # Down-sweep
    d = 1
    while d < padded_n:
        offset //= 2
        if tid < d:
            ai = offset * (2 * tid + 1) - 1
            bi = offset * (2 * tid + 2) - 1
            t = shared[ai]
            shared[ai] = shared[bi]
            shared[bi] += t
        cuda.syncthreads()
        d *= 2

    # Convert exclusive -> inclusive and write out
    if 2 * tid < n:
        val = shared[2 * tid] + x[2 * tid]
        out[2 * tid] = val
    if 2 * tid + 1 < n:
        val = shared[2 * tid + 1] + x[2 * tid + 1]
        out[2 * tid + 1] = val


def scan_blelloch_work_efficient(x, out):
    """Host wrapper: single-block work-efficient (Blelloch) inclusive scan."""
    n = x.shape[0]
    padded_n = 1
    while padded_n < n:
        padded_n *= 2
    threads = padded_n // 2

    d_x = cuda.to_device(x.astype(np.float32))
    d_out = cuda.device_array_like(out.astype(np.float32))
    _scan_blelloch_kernel[1, threads](d_x, d_out, n, padded_n)
    cuda.synchronize()
    d_out.copy_to_host(out)
    return out


# --- Task 22: Histogram with Shared-Memory Privatization ---
@cuda.jit
def _histogram_privatized_kernel(data, bins_out, num_bins):
    s_bins = cuda.shared.array(shape=(256,), dtype=int32)
    tid = cuda.threadIdx.x
    if tid < num_bins:
        s_bins[tid] = 0
    cuda.syncthreads()

    i = cuda.grid(1)
    stride = cuda.gridsize(1)
    n = data.shape[0]
    for idx in range(i, n, stride):
        b = data[idx]
        if 0 <= b < num_bins:
            cuda.atomic.add(s_bins, b, 1)
    cuda.syncthreads()

    if tid < num_bins:
        cuda.atomic.add(bins_out, tid, s_bins[tid])


def histogram_privatized(data, bins_out, num_bins=256):
    """Host wrapper: 256-bin histogram using block-private shared bins."""
    n = data.shape[0]
    bpg = min(64, (n + TPB - 1) // TPB)
    d_data = cuda.to_device(data.astype(np.int32))
    d_bins = cuda.to_device(np.zeros(num_bins, dtype=np.int32))
    _histogram_privatized_kernel[bpg, TPB](d_data, d_bins, num_bins)
    cuda.synchronize()
    d_bins.copy_to_host(bins_out)
    return bins_out


# --- Task 23: Race-Condition Safe Dynamic Work Queue ---
@cuda.jit
def _dynamic_work_queue_kernel(queue_in, queue_out, counter, threshold):
    i = cuda.grid(1)
    if i < queue_in.shape[0]:
        val = queue_in[i]
        if val > threshold:
            slot = cuda.atomic.add(counter, 0, 1)
            queue_out[slot] = val


def dynamic_work_queue(queue_in, queue_out, counter, threshold=0.5):
    """Host wrapper: compacts elements > threshold into queue_out via atomic tail claim."""
    n = queue_in.shape[0]
    bpg = (n + TPB - 1) // TPB
    d_in = cuda.to_device(queue_in.astype(np.float32))
    d_out = cuda.device_array_like(queue_out.astype(np.float32))
    d_counter = cuda.to_device(np.zeros(1, dtype=np.int32))

    _dynamic_work_queue_kernel[bpg, TPB](d_in, d_out, d_counter, threshold)
    cuda.synchronize()

    d_out.copy_to_host(queue_out)
    d_counter.copy_to_host(counter)
    return queue_out, counter


# --- Task 24: Float Atomic Add Emulation via Compare-And-Swap ---
@cuda.jit
def _atomic_cas_float_add_kernel(arr_as_int32, index, val, n_threads):
    i = cuda.grid(1)
    if i >= n_threads:
        return
    old = arr_as_int32[index]
    while True:
        assumed = old
        old_float = cuda.int_as_float(assumed)
        new_bits = cuda.float_as_int(old_float + val)
        old = cuda.atomic.cas(arr_as_int32, index, assumed, new_bits)
        if old == assumed:
            break


def atomic_cas_float_add(arr, index, val, n_threads=1024):
    """Host wrapper: n_threads concurrently add `val` to arr[index] via CAS loop.

    `arr` is a float32 host array; the CAS loop operates on its int32 bit
    representation since numba's cuda.atomic.cas only supports integer types.
    """
    arr_int = arr.view(np.int32).copy()
    d_arr = cuda.to_device(arr_int)

    bpg = (n_threads + TPB - 1) // TPB
    _atomic_cas_float_add_kernel[bpg, TPB](d_arr, index, val, n_threads)
    cuda.synchronize()

    d_arr.copy_to_host(arr_int)
    arr[:] = arr_int.view(np.float32)
    return arr


# --- Task 25: Sparse Vector Dot Product with Atomic Accumulation ---
@cuda.jit
def _sparse_dot_kernel(indices, values, dense_vec, result):
    i = cuda.grid(1)
    stride = cuda.gridsize(1)
    nnz = indices.shape[0]
    for k in range(i, nnz, stride):
        cuda.atomic.add(result, 0, values[k] * dense_vec[indices[k]])


def sparse_dot_product_atomic(indices, values, dense_vec, result):
    """Host wrapper: dot product of a sparse vector (indices/values) with dense_vec."""
    nnz = indices.shape[0]
    bpg = min(64, (nnz + TPB - 1) // TPB)
    d_idx = cuda.to_device(indices.astype(np.int32))
    d_val = cuda.to_device(values.astype(np.float32))
    d_dense = cuda.to_device(dense_vec.astype(np.float32))
    d_result = cuda.to_device(np.zeros(1, dtype=np.float32))

    _sparse_dot_kernel[bpg, TPB](d_idx, d_val, d_dense, d_result)
    cuda.synchronize()
    d_result.copy_to_host(result)
    return result


In [ ]:
# --- Module 3 Verification ---
print("Module 3: Warp Intrinsics & Atomics — verification")

# Task 18
n = 100_000
in_arr = np.random.rand(n).astype(np.float32)
out_arr = np.zeros(1, dtype=np.float32)
warp_reduce_sum_shfl(in_arr, out_arr)
np.testing.assert_allclose(out_arr[0], np.sum(in_arr), rtol=1e-3)
print("[PASS] Task 18: warp_reduce_sum_shfl")

# Task 19
n = 256
data = np.random.rand(n).astype(np.float32) - 0.5
mask_out = np.zeros(n // WARP, dtype=np.uint32)
warp_vote_analyzer(data, mask_out)
for w in range(n // WARP):
    expected = 0
    for lane in range(WARP):
        if data[w * WARP + lane] > 0:
            expected |= (1 << lane)
    assert mask_out[w] == np.uint32(expected)
print("[PASS] Task 19: warp_vote_analyzer")

# Task 20
n = 1000
x = np.random.rand(n).astype(np.float32)
out = np.empty_like(x)
scan_hillis_steele(x, out)
np.testing.assert_allclose(out, np.cumsum(x), atol=1e-3)
print("[PASS] Task 20: scan_hillis_steele")

# Task 21
n = 1000
x = np.random.rand(n).astype(np.float32)
out = np.empty_like(x)
scan_blelloch_work_efficient(x, out)
np.testing.assert_allclose(out, np.cumsum(x), atol=1e-3)
print("[PASS] Task 21: scan_blelloch_work_efficient")

# Task 22
n = 1_000_000
data = np.random.randint(0, 256, size=n).astype(np.int32)
bins_out = np.zeros(256, dtype=np.int32)
histogram_privatized(data, bins_out, 256)
ref, _ = np.histogram(data, bins=256, range=(0, 256))
np.testing.assert_equal(bins_out, ref)
print("[PASS] Task 22: histogram_privatized")

# Task 23
n = 10_000
queue_in = np.random.rand(n).astype(np.float32)
queue_out = np.zeros(n, dtype=np.float32)
counter = np.zeros(1, dtype=np.int32)
dynamic_work_queue(queue_in, queue_out, counter, threshold=0.5)
expected = queue_in[queue_in > 0.5]
assert counter[0] == expected.shape[0]
assert np.isclose(np.sort(queue_out[: counter[0]]).sum(), expected.sum())
print("[PASS] Task 23: dynamic_work_queue")

# Task 24
arr = np.zeros(1, dtype=np.float32)
atomic_cas_float_add(arr, 0, 1.0, n_threads=1024)
assert arr[0] == 1024.0
print("[PASS] Task 24: atomic_cas_float_add")

# Task 25
nnz = 5000
dense_dim = 10000
indices = np.random.randint(0, dense_dim, size=nnz).astype(np.int32)
values = np.random.rand(nnz).astype(np.float32)
dense_vec = np.random.rand(dense_dim).astype(np.float32)
result = np.zeros(1, dtype=np.float32)
sparse_dot_product_atomic(indices, values, dense_vec, result)
ref = np.dot(dense_vec[indices], values)
np.testing.assert_allclose(result[0], ref, rtol=1e-3)
print("[PASS] Task 25: sparse_dot_product_atomic")

print("\nModule 3: all 8 tasks passed.")


## Module 4: Scientific Computing & Numerical Methods (Tasks 26–34)

In [ ]:
# =============================================================================
# Module 4: Scientific Computing & Numerical Methods (Tasks 26-34)
# =============================================================================
from numba import cuda
import numpy as np
import cupy as cp
import math


# --- Task 26: 2D Heat Diffusion Equation Solver (Jacobi Stencil) ---
@cuda.jit
def _heat_diffusion_2d_kernel(u_curr, u_next, alpha, dt, dx, dy):
    i, j = cuda.grid(2)
    M, N = u_curr.shape
    if 0 < i < M - 1 and 0 < j < N - 1:
        lap = (
            (u_curr[i + 1, j] - 2 * u_curr[i, j] + u_curr[i - 1, j]) / (dx * dx)
            + (u_curr[i, j + 1] - 2 * u_curr[i, j] + u_curr[i, j - 1]) / (dy * dy)
        )
        u_next[i, j] = u_curr[i, j] + alpha * dt * lap
    elif i < M and j < N:
        u_next[i, j] = u_curr[i, j]  # Dirichlet: hold boundary fixed


def heat_diffusion_2d_step(u_curr, u_next, alpha, dt, dx, dy):
    """Host wrapper: one Jacobi time-step of the 2D heat equation."""
    M, N = u_curr.shape
    threads = (16, 16)
    blocks = ((M + 15) // 16, (N + 15) // 16)
    d_curr = cuda.to_device(u_curr.astype(np.float64))
    d_next = cuda.device_array_like(u_next.astype(np.float64))
    _heat_diffusion_2d_kernel[blocks, threads](d_curr, d_next, alpha, dt, dx, dy)
    cuda.synchronize()
    d_next.copy_to_host(u_next)
    return u_next


# --- Task 27: 1D Wave Equation FDTD Solver ---
@cuda.jit
def _wave_equation_1d_kernel(u_prev, u_curr, u_next, c, dt, dx):
    i = cuda.grid(1)
    n = u_curr.shape[0]
    if 0 < i < n - 1:
        courant2 = (c * dt / dx) ** 2
        u_next[i] = (
            2 * u_curr[i] - u_prev[i]
            + courant2 * (u_curr[i + 1] - 2 * u_curr[i] + u_curr[i - 1])
        )
    elif i < n:
        u_next[i] = u_curr[i]  # fixed boundary


def wave_equation_1d_step(u_prev, u_curr, u_next, c, dt, dx):
    """Host wrapper: one leapfrog time-step of the 1D wave equation."""
    n = u_curr.shape[0]
    bpg = (n + TPB - 1) // TPB
    d_prev = cuda.to_device(u_prev.astype(np.float64))
    d_curr = cuda.to_device(u_curr.astype(np.float64))
    d_next = cuda.device_array_like(u_next.astype(np.float64))
    _wave_equation_1d_kernel[bpg, TPB](d_prev, d_curr, d_next, c, dt, dx)
    cuda.synchronize()
    d_next.copy_to_host(u_next)
    return u_next


# --- Task 28: N-Body Gravitational Interaction (All-Pairs) ---
NBODY_BLOCK = 128


@cuda.jit
def _nbody_kernel(positions, velocities, masses, dt, softening):
    i = cuda.grid(1)
    n = positions.shape[0]
    sx = cuda.shared.array((NBODY_BLOCK,), dtype=np.float64)
    sy = cuda.shared.array((NBODY_BLOCK,), dtype=np.float64)
    sz = cuda.shared.array((NBODY_BLOCK,), dtype=np.float64)
    sm = cuda.shared.array((NBODY_BLOCK,), dtype=np.float64)

    px = positions[i, 0] if i < n else 0.0
    py = positions[i, 1] if i < n else 0.0
    pz = positions[i, 2] if i < n else 0.0
    ax, ay, az = 0.0, 0.0, 0.0

    n_tiles = (n + NBODY_BLOCK - 1) // NBODY_BLOCK
    for t in range(n_tiles):
        idx = t * NBODY_BLOCK + cuda.threadIdx.x
        if idx < n:
            sx[cuda.threadIdx.x] = positions[idx, 0]
            sy[cuda.threadIdx.x] = positions[idx, 1]
            sz[cuda.threadIdx.x] = positions[idx, 2]
            sm[cuda.threadIdx.x] = masses[idx]
        else:
            sm[cuda.threadIdx.x] = 0.0
        cuda.syncthreads()

        for k in range(NBODY_BLOCK):
            dx_ = sx[k] - px
            dy_ = sy[k] - py
            dz_ = sz[k] - pz
            r2 = dx_ * dx_ + dy_ * dy_ + dz_ * dz_ + softening * softening
            inv_r3 = sm[k] / (r2 * math.sqrt(r2))
            ax += dx_ * inv_r3
            ay += dy_ * inv_r3
            az += dz_ * inv_r3
        cuda.syncthreads()

    if i < n:
        velocities[i, 0] += ax * dt
        velocities[i, 1] += ay * dt
        velocities[i, 2] += az * dt


def nbody_all_pairs(positions, velocities, masses, dt, softening):
    """Host wrapper: one velocity update step for N-body gravity (all-pairs, tiled)."""
    n = positions.shape[0]
    bpg = (n + NBODY_BLOCK - 1) // NBODY_BLOCK
    d_pos = cuda.to_device(positions.astype(np.float64))
    d_vel = cuda.to_device(velocities.astype(np.float64))
    d_mass = cuda.to_device(masses.astype(np.float64))
    _nbody_kernel[bpg, NBODY_BLOCK](d_pos, d_vel, d_mass, dt, softening)
    cuda.synchronize()
    d_vel.copy_to_host(velocities)
    return velocities


# --- Task 29: 2D Separable Gaussian Blur Filter ---
@cuda.jit
def _gaussian_pass_kernel(img_in, img_out, kernel_1d, radius, horizontal):
    y, x = cuda.grid(2)
    H, W = img_in.shape
    if y < H and x < W:
        acc = 0.0
        for k in range(-radius, radius + 1):
            # scipy.ndimage default 'reflect' mode: (d c b a | a b c d | d c b a)
            if horizontal:
                sx = x + k
                if sx < 0:
                    sx = -sx - 1
                elif sx >= W:
                    sx = 2 * W - sx - 1
                acc += img_in[y, sx] * kernel_1d[k + radius]
            else:
                sy = y + k
                if sy < 0:
                    sy = -sy - 1
                elif sy >= H:
                    sy = 2 * H - sy - 1
                acc += img_in[sy, x] * kernel_1d[k + radius]
        img_out[y, x] = acc


def gaussian_blur_2d_separable(img_in, img_out, temp_buf, kernel_1d):
    """Host wrapper: two-pass separable Gaussian blur (horizontal, then vertical)."""
    H, W = img_in.shape
    radius = kernel_1d.shape[0] // 2
    threads = (16, 16)
    blocks = ((H + 15) // 16, (W + 15) // 16)

    d_in = cuda.to_device(img_in.astype(np.float32))
    d_temp = cuda.device_array_like(temp_buf.astype(np.float32))
    d_out = cuda.device_array_like(img_out.astype(np.float32))
    d_kernel = cuda.to_device(kernel_1d.astype(np.float32))

    _gaussian_pass_kernel[blocks, threads](d_in, d_temp, d_kernel, radius, True)
    cuda.synchronize()
    _gaussian_pass_kernel[blocks, threads](d_temp, d_out, d_kernel, radius, False)
    cuda.synchronize()

    d_temp.copy_to_host(temp_buf)
    d_out.copy_to_host(img_out)
    return img_out


# --- Task 30: 2D Sobel Filter Edge Detection ---
@cuda.jit
def _sobel_kernel(img_in, magnitude, orientation):
    y, x = cuda.grid(2)
    H, W = img_in.shape
    if y < H and x < W:
        ym = max(y - 1, 0)
        yp = min(y + 1, H - 1)
        xm = max(x - 1, 0)
        xp = min(x + 1, W - 1)
        gx = (
            -img_in[ym, xm] + img_in[ym, xp]
            - 2 * img_in[y, xm] + 2 * img_in[y, xp]
            - img_in[yp, xm] + img_in[yp, xp]
        )
        gy = (
            -img_in[ym, xm] - 2 * img_in[ym, x] - img_in[ym, xp]
            + img_in[yp, xm] + 2 * img_in[yp, x] + img_in[yp, xp]
        )
        magnitude[y, x] = math.sqrt(gx * gx + gy * gy)
        orientation[y, x] = math.atan2(gy, gx)


def sobel_edge_detector(img_in, magnitude, orientation):
    """Host wrapper: Sobel gradient magnitude + orientation, clamped (replicate) borders."""
    H, W = img_in.shape
    threads = (16, 16)
    blocks = ((H + 15) // 16, (W + 15) // 16)
    d_in = cuda.to_device(img_in.astype(np.float32))
    d_mag = cuda.device_array_like(magnitude.astype(np.float32))
    d_ori = cuda.device_array_like(orientation.astype(np.float32))
    _sobel_kernel[blocks, threads](d_in, d_mag, d_ori)
    cuda.synchronize()
    d_mag.copy_to_host(magnitude)
    d_ori.copy_to_host(orientation)
    return magnitude, orientation


# --- Task 31: Monte Carlo Pi Estimation via CuPy Random ---
def monte_carlo_pi_cupy(num_samples, batch=50_000_000):
    """Estimates Pi by sampling uniform points in the unit square (CuPy), batched."""
    inside = 0
    remaining = num_samples
    while remaining > 0:
        m = min(batch, remaining)
        x = cp.random.uniform(0.0, 1.0, size=m, dtype=cp.float32)
        y = cp.random.uniform(0.0, 1.0, size=m, dtype=cp.float32)
        inside += int(cp.count_nonzero(x * x + y * y <= 1.0))
        remaining -= m
    return 4.0 * inside / num_samples


# --- Task 32: K-Means Clustering Assignment Step ---
@cuda.jit
def _kmeans_assign_kernel(points, centroids, labels, distances):
    i = cuda.grid(1)
    n, d = points.shape
    k = centroids.shape[0]
    if i < n:
        best_dist = 1e38
        best_k = 0
        for c in range(k):
            dist = 0.0
            for dim in range(d):
                diff = points[i, dim] - centroids[c, dim]
                dist += diff * diff
            if dist < best_dist:
                best_dist = dist
                best_k = c
        labels[i] = best_k
        distances[i] = best_dist


def kmeans_assign_clusters(points, centroids, labels, distances):
    """Host wrapper: assigns each point to its nearest centroid (squared Euclidean)."""
    n = points.shape[0]
    bpg = (n + TPB - 1) // TPB
    d_pts = cuda.to_device(points.astype(np.float32))
    d_cent = cuda.to_device(centroids.astype(np.float32))
    d_labels = cuda.device_array_like(labels.astype(np.int32))
    d_dist = cuda.device_array_like(distances.astype(np.float32))
    _kmeans_assign_kernel[bpg, TPB](d_pts, d_cent, d_labels, d_dist)
    cuda.synchronize()
    d_labels.copy_to_host(labels)
    d_dist.copy_to_host(distances)
    return labels, distances


# --- Task 33: FFT Radix-2 Cooley-Tukey Butterfly Stage ---
@cuda.jit
def _fft_butterfly_kernel(data_re, data_im, stage, N):
    i = cuda.grid(1)
    half = N // 2
    if i < half:
        span = 1 << (stage - 1)          # half-size of a butterfly group
        group = i // span
        pos = i % span
        idx1 = group * 2 * span + pos
        idx2 = idx1 + span

        angle = -2.0 * math.pi * pos / (1 << stage)
        wr = math.cos(angle)
        wi = math.sin(angle)

        ar, ai = data_re[idx1], data_im[idx1]
        br, bi = data_re[idx2], data_im[idx2]
        tr = br * wr - bi * wi
        ti = br * wi + bi * wr

        data_re[idx1] = ar + tr
        data_im[idx1] = ai + ti
        data_re[idx2] = ar - tr
        data_im[idx2] = ai - ti


def fft_radix2_butterfly_stage(data_re, data_im, stage, N):
    """Host wrapper: one butterfly stage (stage = 1..log2(N)) of radix-2 DIT FFT, in-place.

    Input must already be in bit-reversed order before stage 1.
    """
    half = N // 2
    bpg = (half + TPB - 1) // TPB
    d_re = cuda.to_device(data_re.astype(np.float64))
    d_im = cuda.to_device(data_im.astype(np.float64))
    _fft_butterfly_kernel[bpg, TPB](d_re, d_im, stage, N)
    cuda.synchronize()
    d_re.copy_to_host(data_re)
    d_im.copy_to_host(data_im)
    return data_re, data_im


# --- Task 34: Mandelbrot Fractal Generation with Escape Time ---
@cuda.jit
def _mandelbrot_kernel(out, xmin, xmax, ymin, ymax, max_iter):
    px, py = cuda.grid(2)
    H, W = out.shape
    if py < H and px < W:
        cx = xmin + (xmax - xmin) * px / W
        cy = ymin + (ymax - ymin) * py / H
        zr, zi = 0.0, 0.0
        it = 0
        while zr * zr + zi * zi <= 4.0 and it < max_iter:
            zr, zi = zr * zr - zi * zi + cx, 2.0 * zr * zi + cy
            it += 1
        out[py, px] = it


def mandelbrot_escape_time(out, xmin, xmax, ymin, ymax, max_iter):
    """Host wrapper: escape-time iteration count per pixel over the complex plane."""
    H, W = out.shape
    threads = (16, 16)
    blocks = ((W + 15) // 16, (H + 15) // 16)
    d_out = cuda.device_array_like(out.astype(np.int32))
    _mandelbrot_kernel[blocks, threads](d_out, xmin, xmax, ymin, ymax, max_iter)
    cuda.synchronize()
    d_out.copy_to_host(out)
    return out


In [ ]:
# --- Module 4 Verification ---
print("Module 4: Scientific Computing & Numerical Methods — verification")


def cpu_jacobi_reference(u, alpha, dt, dx, dy):
    u_next = u.copy()
    lap = (
        (u[2:, 1:-1] - 2 * u[1:-1, 1:-1] + u[:-2, 1:-1]) / dx**2
        + (u[1:-1, 2:] - 2 * u[1:-1, 1:-1] + u[1:-1, :-2]) / dy**2
    )
    u_next[1:-1, 1:-1] = u[1:-1, 1:-1] + alpha * dt * lap
    return u_next


# Task 26
M, N = 50, 50
u = np.random.rand(M, N).astype(np.float64)
u_next = np.empty_like(u)
heat_diffusion_2d_step(u, u_next, alpha=0.1, dt=0.01, dx=1.0, dy=1.0)
ref = cpu_jacobi_reference(u, 0.1, 0.01, 1.0, 1.0)
np.testing.assert_allclose(u_next[1:-1, 1:-1], ref[1:-1, 1:-1], atol=1e-5)
print("[PASS] Task 26: heat_diffusion_2d_step")


def cpu_wave_step(u_prev, u_curr, c, dt, dx):
    u_next = u_curr.copy()
    courant2 = (c * dt / dx) ** 2
    u_next[1:-1] = (
        2 * u_curr[1:-1] - u_prev[1:-1]
        + courant2 * (u_curr[2:] - 2 * u_curr[1:-1] + u_curr[:-2])
    )
    return u_next


# Task 27
n = 500
u_prev = np.sin(np.linspace(0, 6, n))
u_curr = np.sin(np.linspace(0.1, 6.1, n))
u_next = np.empty_like(u_curr)
wave_equation_1d_step(u_prev, u_curr, u_next, c=1.0, dt=0.01, dx=0.1)
ref = cpu_wave_step(u_prev, u_curr, 1.0, 0.01, 0.1)
np.testing.assert_allclose(u_next, ref, atol=1e-4)
print("[PASS] Task 27: wave_equation_1d_step")


def cpu_nbody_reference(positions, velocities, masses, dt, softening):
    n = positions.shape[0]
    vel = velocities.copy()
    for i in range(n):
        acc = np.zeros(3)
        for j in range(n):
            d = positions[j] - positions[i]
            r2 = d @ d + softening**2
            acc += masses[j] * d / (r2 * np.sqrt(r2))
        vel[i] += acc * dt
    return vel


# Task 28
n = 64
positions = np.random.rand(n, 3).astype(np.float64)
velocities = np.zeros((n, 3), dtype=np.float64)
masses = np.random.rand(n).astype(np.float64) + 0.5
vel_before = velocities.copy()
nbody_all_pairs(positions, velocities, masses, dt=0.01, softening=0.1)
ref = cpu_nbody_reference(positions, vel_before, masses, 0.01, 0.1)
np.testing.assert_allclose(velocities, ref, atol=1e-3)
print("[PASS] Task 28: nbody_all_pairs")

# Task 29 (scipy default: sigma=2.0, truncate=4.0 -> radius 8, mode='reflect')
H, W = 128, 128
img_in = np.random.rand(H, W).astype(np.float32)
sigma = 2.0
radius = int(4.0 * sigma + 0.5)
xs = np.arange(-radius, radius + 1)
kernel_1d = np.exp(-(xs**2) / (2 * sigma**2))
kernel_1d = (kernel_1d / kernel_1d.sum()).astype(np.float32)
img_out = np.empty_like(img_in)
temp_buf = np.empty_like(img_in)
gaussian_blur_2d_separable(img_in, img_out, temp_buf, kernel_1d)
ref = scipy.ndimage.gaussian_filter(img_in, sigma=2.0)
np.testing.assert_allclose(img_out, ref, atol=1e-3)
print("[PASS] Task 29: gaussian_blur_2d_separable")


def cpu_sobel(img):
    p = np.pad(img, 1, mode="edge")
    gx = (-p[:-2, :-2] + p[:-2, 2:] - 2 * p[1:-1, :-2] + 2 * p[1:-1, 2:] - p[2:, :-2] + p[2:, 2:])
    gy = (-p[:-2, :-2] - 2 * p[:-2, 1:-1] - p[:-2, 2:] + p[2:, :-2] + 2 * p[2:, 1:-1] + p[2:, 2:])
    return np.sqrt(gx**2 + gy**2), np.arctan2(gy, gx)


# Task 30
H, W = 64, 64
img_in = np.random.rand(H, W).astype(np.float32)
magnitude = np.empty((H, W), dtype=np.float32)
orientation = np.empty((H, W), dtype=np.float32)
sobel_edge_detector(img_in, magnitude, orientation)
ref_mag, ref_ori = cpu_sobel(img_in)
np.testing.assert_allclose(magnitude, ref_mag, atol=1e-3)
np.testing.assert_allclose(orientation, ref_ori, atol=1e-3)
print("[PASS] Task 30: sobel_edge_detector")

# Task 31
pi_est = monte_carlo_pi_cupy(100_000_000)
assert abs(pi_est - np.pi) < 1e-3
print(f"[PASS] Task 31: monte_carlo_pi_cupy (pi={pi_est:.5f})")


def cpu_kmeans_assign(points, centroids):
    d2 = ((points[:, None, :] - centroids[None, :, :]) ** 2).sum(axis=2)
    return np.argmin(d2, axis=1)


# Task 32
n, d, k = 2000, 4, 8
points = np.random.rand(n, d).astype(np.float32)
centroids = np.random.rand(k, d).astype(np.float32)
labels = np.empty(n, dtype=np.int32)
distances = np.empty(n, dtype=np.float32)
kmeans_assign_clusters(points, centroids, labels, distances)
ref = cpu_kmeans_assign(points, centroids)
np.testing.assert_equal(labels, ref)
print("[PASS] Task 32: kmeans_assign_clusters")

# Task 33
N = 16
x = np.random.rand(N) + 1j * np.random.rand(N)
# bit-reversal permutation to match standard DIT ordering
bits = int(np.log2(N))
rev = np.array([int(f"{i:0{bits}b}"[::-1], 2) for i in range(N)])
data_re = x.real[rev].copy()
data_im = x.imag[rev].copy()
for stage in range(1, bits + 1):
    fft_radix2_butterfly_stage(data_re, data_im, stage, N)
result = data_re + 1j * data_im
ref = np.fft.fft(x)
np.testing.assert_allclose(result, ref, atol=1e-6)
print("[PASS] Task 33: fft_radix2_butterfly_stage (matches scipy/np fft)")


def cpu_mandelbrot(xmin, xmax, ymin, ymax, w, h, max_iter):
    out = np.zeros((h, w), dtype=np.int32)
    for py in range(h):
        cy = ymin + (ymax - ymin) * py / h
        for px in range(w):
            cx = xmin + (xmax - xmin) * px / w
            z = 0j
            c = complex(cx, cy)
            it = 0
            while abs(z) <= 2.0 and it < max_iter:
                z = z * z + c
                it += 1
            out[py, px] = it
    return out


# Task 34 (small grid for a fast CPU reference comparison)
H, W = 40, 40
out = np.zeros((H, W), dtype=np.int32)
mandelbrot_escape_time(out, -2.0, 1.0, -1.5, 1.5, 100)
ref = cpu_mandelbrot(-2.0, 1.0, -1.5, 1.5, W, H, 100)
np.testing.assert_equal(out, ref)
print("[PASS] Task 34: mandelbrot_escape_time")

print("\nModule 4: all 9 tasks passed.")


## Module 5: Streams, Concurrency & CuPy Interop (Tasks 35–42)

In [ ]:
# =============================================================================
# Module 5: Streams, Concurrency & CuPy Interop (Tasks 35-42)
# =============================================================================
from numba import cuda
import numpy as np
import cupy as cp
import time


# --- Task 35: Asynchronous Dual-Stream Vector Processing Pipeline ---
@cuda.jit
def _double_kernel(d_arr):
    i = cuda.grid(1)
    if i < d_arr.shape[0]:
        d_arr[i] = d_arr[i] * 2.0


def pipeline_dual_stream(h_in, h_out, chunk_size):
    """Overlaps H2D transfer of chunk 1 with compute on chunk 0 using 2 streams."""
    n = h_in.shape[0]
    stream0 = cuda.stream()
    stream1 = cuda.stream()

    c0 = h_in[:chunk_size]
    c1 = h_in[chunk_size:]

    d0 = cuda.to_device(c0, stream=stream0)
    d1 = cuda.to_device(c1, stream=stream1)  # overlaps with chunk-0 compute below

    bpg0 = (c0.shape[0] + TPB - 1) // TPB
    _double_kernel[bpg0, TPB, stream0](d0)

    bpg1 = (c1.shape[0] + TPB - 1) // TPB
    _double_kernel[bpg1, TPB, stream1](d1)

    d0.copy_to_host(h_out[:chunk_size], stream=stream0)
    d1.copy_to_host(h_out[chunk_size:], stream=stream1)
    stream0.synchronize()
    stream1.synchronize()
    return h_out


# --- Task 36: Multi-Stream Matrix-Vector Multiply with Overlapped Transfers ---
@cuda.jit
def _gemv_kernel(matrix, vec, out_vec):
    row = cuda.grid(1)
    M, N = matrix.shape
    if row < M:
        acc = 0.0
        for j in range(N):
            acc += matrix[row, j] * vec[j]
        out_vec[row] = acc


def multistream_gemv(matrix, vectors, out_vectors, num_streams=4):
    """Processes a batch of vectors through matrix @ v using a round-robin stream pool."""
    M, N = matrix.shape
    B = vectors.shape[0]
    streams = [cuda.stream() for _ in range(num_streams)]
    d_matrix = cuda.to_device(matrix.astype(np.float32))
    bpg = (M + TPB - 1) // TPB

    d_out_list = []
    for b in range(B):
        s = streams[b % num_streams]
        d_vec = cuda.to_device(vectors[b].astype(np.float32), stream=s)
        d_out = cuda.device_array(M, dtype=np.float32, stream=s)
        _gemv_kernel[bpg, TPB, s](d_matrix, d_vec, d_out)
        d_out.copy_to_host(out_vectors[b], stream=s)
        d_out_list.append(d_out)

    for s in streams:
        s.synchronize()
    return out_vectors


# --- Task 37: CuPy RawKernel — Custom Vector Sinc Function ---
_sinc_kernel_src = r"""
extern "C" __global__
void sinc_kernel(const float* x, float* out, int N) {
    int i = blockDim.x * blockIdx.x + threadIdx.x;
    if (i < N) {
        float xi = x[i];
        if (xi == 0.0f) {
            out[i] = 1.0f;
        } else {
            float px = 3.14159265358979323846f * xi;
            out[i] = sinf(px) / px;
        }
    }
}
"""
_sinc_kernel = cp.RawKernel(_sinc_kernel_src, "sinc_kernel")


def cupy_rawkernel_sinc(x, out):
    """Normalized sinc(x) = sin(pi*x)/(pi*x), sinc(0) = 1, via a RawKernel."""
    d_x = cp.asarray(x, dtype=cp.float32)
    d_out = cp.empty_like(d_x)
    n = d_x.shape[0]
    tpb, bpg = 256, (n + 255) // 256
    _sinc_kernel((bpg,), (tpb,), (d_x, d_out, np.int32(n)))
    cp.cuda.Stream.null.synchronize()
    out[:] = cp.asnumpy(d_out)
    return out


# --- Task 38: CuPy RawModule — Multi-Function Arithmetic Kernel ---
_arith_module_src = r"""
extern "C" __global__
void vector_add(const float* a, const float* b, float* out, int N) {
    int i = blockDim.x * blockIdx.x + threadIdx.x;
    if (i < N) out[i] = a[i] + b[i];
}
extern "C" __global__
void vector_sub(const float* a, const float* b, float* out, int N) {
    int i = blockDim.x * blockIdx.x + threadIdx.x;
    if (i < N) out[i] = a[i] - b[i];
}
extern "C" __global__
void vector_mul(const float* a, const float* b, float* out, int N) {
    int i = blockDim.x * blockIdx.x + threadIdx.x;
    if (i < N) out[i] = a[i] * b[i];
}
"""
_arith_module = cp.RawModule(code=_arith_module_src)
_k_add = _arith_module.get_function("vector_add")
_k_sub = _arith_module.get_function("vector_sub")
_k_mul = _arith_module.get_function("vector_mul")


def cupy_rawmodule_arithmetic(a, b, res_add, res_sub, res_mul):
    """Compiles one RawModule with add/sub/mul kernels and runs all three."""
    d_a = cp.asarray(a, dtype=cp.float32)
    d_b = cp.asarray(b, dtype=cp.float32)
    n = d_a.shape[0]
    tpb, bpg = 256, (n + 255) // 256

    d_add = cp.empty_like(d_a)
    d_sub = cp.empty_like(d_a)
    d_mul = cp.empty_like(d_a)
    _k_add((bpg,), (tpb,), (d_a, d_b, d_add, np.int32(n)))
    _k_sub((bpg,), (tpb,), (d_a, d_b, d_sub, np.int32(n)))
    _k_mul((bpg,), (tpb,), (d_a, d_b, d_mul, np.int32(n)))
    cp.cuda.Stream.null.synchronize()

    res_add[:] = cp.asnumpy(d_add)
    res_sub[:] = cp.asnumpy(d_sub)
    res_mul[:] = cp.asnumpy(d_mul)
    return res_add, res_sub, res_mul


# --- Task 39: CUDA Event Timing & Kernel Profiling Wrapper ---
def cuda_event_profiler(kernel_func, args, num_warmup=3, num_repeats=10):
    """Times kernel_func(*args) with CUDA events; returns (mean_ms, std_ms)."""
    for _ in range(num_warmup):
        kernel_func(*args)
    cuda.synchronize()

    times = []
    for _ in range(num_repeats):
        start = cuda.event()
        end = cuda.event()
        start.record()
        kernel_func(*args)
        end.record()
        end.synchronize()
        times.append(cuda.event_elapsed_time(start, end))

    times = np.array(times)
    return float(times.mean()), float(times.std())


# --- Task 40: CUDA Graph Creation and Instantiation ---
def cuda_graph_capture_exec(a, b, c, num_iterations):
    """Captures add+scale kernel launches into a CUDA Graph and replays it."""
    d_a = cp.asarray(a, dtype=cp.float32)
    d_b = cp.asarray(b, dtype=cp.float32)
    d_c = cp.asarray(c, dtype=cp.float32)

    stream = cp.cuda.Stream(non_blocking=True)
    with stream:
        stream.begin_capture()
        cp.add(d_a, d_b, out=d_c)          # no allocations inside capture
        cp.multiply(d_c, 1.0001, out=d_c)
        graph = stream.end_capture()

    for _ in range(num_iterations):
        graph.launch(stream=stream)
    stream.synchronize()

    return cp.asnumpy(d_c)


# --- Task 41: Unified Memory Zero-Copy Access Benchmark ---
def benchmark_unified_memory(size_bytes):
    """Benchmarks first-touch page faults vs explicit device buffers for managed memory."""
    n = size_bytes // 8  # float64 elements

    # Managed (Unified) memory
    mem = cuda.managed_array(n, dtype=np.float64)
    mem[:] = 1.0  # first touch on host

    start = time.perf_counter()
    mem[:] = mem * 2.0  # host modification
    first_touch_s = time.perf_counter() - start

    @cuda.jit
    def _gpu_read_kernel(arr):
        i = cuda.grid(1)
        if i < arr.shape[0]:
            arr[i] += 1.0

    bpg = (n + TPB - 1) // TPB
    start = time.perf_counter()
    _gpu_read_kernel[bpg, TPB](mem)
    cuda.synchronize()
    gpu_access_s = time.perf_counter() - start

    consistent = np.allclose(mem, 3.0)
    return {
        "first_touch_s": first_touch_s,
        "gpu_access_s": gpu_access_s,
        "consistent": bool(consistent),
    }


# --- Task 42: Inter-Kernel Synchronization via Cooperative Groups ---
_coop_kernel_src = r"""
#include <cooperative_groups.h>
namespace cg = cooperative_groups;

extern "C" __global__
void grid_barrier_kernel(float* arr, int iterations, int N) {
    cg::grid_group grid = cg::this_grid();
    int i = blockDim.x * blockIdx.x + threadIdx.x;
    for (int it = 0; it < iterations; it++) {
        if (i < N) {
            arr[i] += 1.0f;
        }
        grid.sync();  // global barrier across ALL blocks in the grid
    }
}
"""


def grid_cooperative_barrier(arr, iterations):
    """Launches a cooperative kernel that globally syncs every block each iteration.

    Requires a device that supports cooperative launch (most modern NVIDIA GPUs);
    falls back with a clear error message otherwise.
    """
    d_arr = cp.asarray(arr, dtype=cp.float32)
    n = d_arr.shape[0]
    tpb = 256
    bpg = (n + tpb - 1) // tpb

    module = cp.RawModule(code=_coop_kernel_src, backend="nvcc",
                          enable_cooperative_groups=True)
    kernel = module.get_function("grid_barrier_kernel")

    try:
        kernel((bpg,), (tpb,), (d_arr, np.int32(iterations), np.int32(n)))
        cp.cuda.Stream.null.synchronize()
    except Exception as e:
        raise RuntimeError(
            "Cooperative grid-sync launch failed (hardware may not support "
            f"cudaLaunchCooperativeKernel at this block/grid size): {e}"
        )

    arr[:] = cp.asnumpy(d_arr)
    return arr


In [ ]:
# --- Module 5 Verification ---
print("Module 5: Streams, Concurrency & CuPy Interop — verification")

# Task 35
n = 100_000
h_in = np.random.rand(n).astype(np.float32)
h_out = np.empty_like(h_in)
pipeline_dual_stream(h_in, h_out, chunk_size=n // 2)
np.testing.assert_allclose(h_out, h_in * 2.0, atol=1e-5)
print("[PASS] Task 35: pipeline_dual_stream")

# Task 36
M, N, B = 64, 48, 10
matrix = np.random.rand(M, N).astype(np.float32)
vectors = np.random.rand(B, N).astype(np.float32)
out_vectors = np.empty((B, M), dtype=np.float32)
multistream_gemv(matrix, vectors, out_vectors, num_streams=4)
np.testing.assert_allclose(out_vectors, vectors @ matrix.T, atol=1e-3)
print("[PASS] Task 36: multistream_gemv")

# Task 37
n = 1000
x = (np.random.rand(n).astype(np.float32) - 0.5) * 10
out = np.empty_like(x)
cupy_rawkernel_sinc(x, out)
ref = np.sinc(x)  # numpy's sinc is already normalized sin(pi x)/(pi x)
np.testing.assert_allclose(out, ref, atol=1e-5)
print("[PASS] Task 37: cupy_rawkernel_sinc")

# Task 38
n = 1000
a = np.random.rand(n).astype(np.float32)
b = np.random.rand(n).astype(np.float32)
res_add = np.empty_like(a)
res_sub = np.empty_like(a)
res_mul = np.empty_like(a)
cupy_rawmodule_arithmetic(a, b, res_add, res_sub, res_mul)
np.testing.assert_allclose(res_add, a + b, atol=1e-5)
np.testing.assert_allclose(res_sub, a - b, atol=1e-5)
np.testing.assert_allclose(res_mul, a * b, atol=1e-5)
print("[PASS] Task 38: cupy_rawmodule_arithmetic")

# Task 39
n = 1_000_000
a = np.random.rand(n).astype(np.float32)
b = np.random.rand(n).astype(np.float32)
out = np.empty_like(a)
mean_ms, std_ms = cuda_event_profiler(vector_add_1d, (a, b, out), num_warmup=3, num_repeats=10)
assert std_ms / max(mean_ms, 1e-6) < 1.0  # sanity: std shouldn't dwarf the mean
print(f"[PASS] Task 39: cuda_event_profiler (mean={mean_ms:.4f}ms, std={std_ms:.4f}ms)")

# Task 40
n = 10_000
a = np.random.rand(n).astype(np.float32)
b = np.random.rand(n).astype(np.float32)
c = np.empty_like(a)
result = cuda_graph_capture_exec(a, b, c, num_iterations=5)
ref = (a + b) * 1.0001
np.testing.assert_allclose(result, ref, atol=1e-4)
print("[PASS] Task 40: cuda_graph_capture_exec")

# Task 41
mem_result = benchmark_unified_memory(16 * 1024 * 1024)
assert mem_result["consistent"]
print(f"[PASS] Task 41: benchmark_unified_memory ({mem_result})")

# Task 42 (requires cooperative-launch hardware support; reports result or explains failure)
try:
    n = 4096
    arr = np.zeros(n, dtype=np.float32)
    grid_cooperative_barrier(arr, iterations=5)
    np.testing.assert_allclose(arr, 5.0)
    print("[PASS] Task 42: grid_cooperative_barrier")
except RuntimeError as e:
    print(f"[SKIP] Task 42: grid_cooperative_barrier — {e}")

print("\nModule 5: tasks 35-41 verified (Task 42 depends on cooperative-launch support).")


## Module 6: Deep Learning & Tensor Primitives (Tasks 43–50)

In [ ]:
# =============================================================================
# Module 6: Deep Learning & Tensor Primitives (Tasks 43-50)
# =============================================================================
from numba import cuda
import numpy as np
import math


# --- Task 43: Numerically Stable Online Softmax ---
@cuda.jit
def _softmax_kernel(logits, probs):
    row = cuda.blockIdx.x
    tid = cuda.threadIdx.x
    V = logits.shape[1]
    block_size = cuda.blockDim.x

    s_max = cuda.shared.array(shape=(256,), dtype=np.float32)
    s_sum = cuda.shared.array(shape=(256,), dtype=np.float32)

    local_max = -3.4e38
    for j in range(tid, V, block_size):
        v = logits[row, j]
        if v > local_max:
            local_max = v
    s_max[tid] = local_max
    cuda.syncthreads()

    stride = block_size // 2
    while stride > 0:
        if tid < stride:
            if s_max[tid + stride] > s_max[tid]:
                s_max[tid] = s_max[tid + stride]
        cuda.syncthreads()
        stride >>= 1
    row_max = s_max[0]
    cuda.syncthreads()

    local_sum = 0.0
    for j in range(tid, V, block_size):
        local_sum += math.exp(logits[row, j] - row_max)
    s_sum[tid] = local_sum
    cuda.syncthreads()

    stride = block_size // 2
    while stride > 0:
        if tid < stride:
            s_sum[tid] += s_sum[tid + stride]
        cuda.syncthreads()
        stride >>= 1
    row_sum = s_sum[0]
    cuda.syncthreads()

    for j in range(tid, V, block_size):
        probs[row, j] = math.exp(logits[row, j] - row_max) / row_sum


def stable_online_softmax(logits, probs):
    """Host wrapper: numerically stable row-wise softmax (one block per row)."""
    Bsz, V = logits.shape
    threads = 256  # power of 2 required by the tree reduction
    d_logits = cuda.to_device(logits.astype(np.float32))
    d_probs = cuda.device_array_like(probs.astype(np.float32))
    _softmax_kernel[Bsz, threads](d_logits, d_probs)
    cuda.synchronize()
    d_probs.copy_to_host(probs)
    return probs


# --- Task 44: Fused Bias-GELU Activation Kernel ---
@cuda.jit
def _fused_bias_gelu_kernel(x, bias, out, D):
    i = cuda.grid(1)
    if i < x.shape[0]:
        y = x[i] + bias[i % D]
        c = 0.7978845608028654  # sqrt(2/pi)
        inner = c * (y + 0.044715 * y * y * y)
        out[i] = 0.5 * y * (1.0 + math.tanh(inner))


def fused_bias_gelu(x, bias, out):
    """Host wrapper: fuses bias-add + GELU activation into a single kernel."""
    n = x.shape[0]
    D = bias.shape[0]
    bpg = (n + TPB - 1) // TPB
    d_x = cuda.to_device(x.astype(np.float32))
    d_bias = cuda.to_device(bias.astype(np.float32))
    d_out = cuda.device_array_like(out.astype(np.float32))
    _fused_bias_gelu_kernel[bpg, TPB](d_x, d_bias, d_out, D)
    cuda.synchronize()
    d_out.copy_to_host(out)
    return out


# --- Task 45: 2D Spatial Max-Pooling with Forward Argmax Tracking ---
@cuda.jit
def _maxpool2d_kernel(x, out, argmax_idx, pool_h, pool_w, stride):
    ow, oh, bc = cuda.grid(3)          # z axis packs (batch, channel)
    B, C, H, W = x.shape
    _, _, OH, OW = out.shape
    if bc < B * C and oh < OH and ow < OW:
        b = bc // C
        c = bc % C
        best_val = -3.4e38
        best_idx = -1
        for ph in range(pool_h):
            for pw in range(pool_w):
                ih = oh * stride + ph
                iw = ow * stride + pw
                if ih < H and iw < W:
                    v = x[b, c, ih, iw]
                    if v > best_val:
                        best_val = v
                        best_idx = ih * W + iw
        out[b, c, oh, ow] = best_val
        argmax_idx[b, c, oh, ow] = best_idx


def maxpool2d_with_argmax(x, out, argmax_idx, pool_h, pool_w, stride):
    """Host wrapper: 2D max-pool over (B, C, H, W), tracking flattened argmax index."""
    B, C, H, W = x.shape
    _, _, OH, OW = out.shape
    threads = (8, 8, 4)
    blocks = (
        (OW + threads[0] - 1) // threads[0],
        (OH + threads[1] - 1) // threads[1],
        (B * C + threads[2] - 1) // threads[2],
    )
    d_x = cuda.to_device(x.astype(np.float32))
    d_out = cuda.device_array_like(out.astype(np.float32))
    d_argmax = cuda.device_array_like(argmax_idx.astype(np.int32))
    _maxpool2d_kernel[blocks, threads](d_x, d_out, d_argmax, pool_h, pool_w, stride)
    cuda.synchronize()
    d_out.copy_to_host(out)
    d_argmax.copy_to_host(argmax_idx)
    return out, argmax_idx


# --- Task 46: 1D Batch Normalization Forward Pass ---
@cuda.jit
def _batch_norm_kernel(x, gamma, beta, eps, out, mean_out, var_out):
    feat = cuda.blockIdx.x
    tid = cuda.threadIdx.x
    B = x.shape[0]
    block_size = cuda.blockDim.x

    s_sum = cuda.shared.array(shape=(256,), dtype=np.float32)
    s_sqsum = cuda.shared.array(shape=(256,), dtype=np.float32)

    local_sum, local_sq = 0.0, 0.0
    for i in range(tid, B, block_size):
        v = x[i, feat]
        local_sum += v
        local_sq += v * v
    s_sum[tid] = local_sum
    s_sqsum[tid] = local_sq
    cuda.syncthreads()

    stride = block_size // 2
    while stride > 0:
        if tid < stride:
            s_sum[tid] += s_sum[tid + stride]
            s_sqsum[tid] += s_sqsum[tid + stride]
        cuda.syncthreads()
        stride >>= 1

    mean = s_sum[0] / B
    var = s_sqsum[0] / B - mean * mean
    if tid == 0:
        mean_out[feat] = mean
        var_out[feat] = var
    cuda.syncthreads()

    inv_std = 1.0 / math.sqrt(var + eps)
    for i in range(tid, B, block_size):
        out[i, feat] = gamma[feat] * (x[i, feat] - mean) * inv_std + beta[feat]


def batch_norm_1d_forward(x, gamma, beta, eps, out, mean_out, var_out):
    """Host wrapper: batch norm across batch dim B, per-feature (one block per feature)."""
    B, D = x.shape
    threads = 256  # power of 2 required by the tree reduction
    d_x = cuda.to_device(x.astype(np.float32))
    d_gamma = cuda.to_device(gamma.astype(np.float32))
    d_beta = cuda.to_device(beta.astype(np.float32))
    d_out = cuda.device_array_like(out.astype(np.float32))
    d_mean = cuda.device_array_like(mean_out.astype(np.float32))
    d_var = cuda.device_array_like(var_out.astype(np.float32))

    _batch_norm_kernel[D, threads](d_x, d_gamma, d_beta, eps, d_out, d_mean, d_var)
    cuda.synchronize()
    d_out.copy_to_host(out)
    d_mean.copy_to_host(mean_out)
    d_var.copy_to_host(var_out)
    return out, mean_out, var_out


# --- Task 47: Layer Normalization using Welford's Algorithm ---
@cuda.jit
def _layer_norm_welford_kernel(x, gamma, beta, eps, out):
    row = cuda.blockIdx.x
    tid = cuda.threadIdx.x
    D = x.shape[1]
    block_size = cuda.blockDim.x

    s_mean = cuda.shared.array(shape=(256,), dtype=np.float32)
    s_m2 = cuda.shared.array(shape=(256,), dtype=np.float32)
    s_count = cuda.shared.array(shape=(256,), dtype=np.int32)

    count, mean, m2 = 0, 0.0, 0.0
    for j in range(tid, D, block_size):
        count += 1
        v = x[row, j]
        delta = v - mean
        mean += delta / count
        m2 += delta * (v - mean)
    s_mean[tid] = mean
    s_m2[tid] = m2
    s_count[tid] = count
    cuda.syncthreads()

    # Tree-merge Welford tuples (Chan's parallel combination formula)
    stride = block_size // 2
    while stride > 0:
        if tid < stride:
            cb, mb, m2b = s_count[tid + stride], s_mean[tid + stride], s_m2[tid + stride]
            ca, ma, m2a = s_count[tid], s_mean[tid], s_m2[tid]
            nc = ca + cb
            if nc > 0:
                delta = mb - ma
                nm = ma + delta * cb / nc if nc > 0 else ma
                nm2 = m2a + m2b + delta * delta * ca * cb / nc
                s_count[tid], s_mean[tid], s_m2[tid] = nc, nm, nm2
        cuda.syncthreads()
        stride >>= 1

    total_mean = s_mean[0]
    total_var = s_m2[0] / D
    cuda.syncthreads()

    inv_std = 1.0 / math.sqrt(total_var + eps)
    for j in range(tid, D, block_size):
        out[row, j] = gamma[j] * (x[row, j] - total_mean) * inv_std + beta[j]


def layer_norm_welford(x, gamma, beta, eps, out):
    """Host wrapper: LayerNorm over hidden dim D, Welford one-pass stats (block per row)."""
    Bsz, D = x.shape
    threads = 256  # power of 2 required by the tree reduction
    d_x = cuda.to_device(x.astype(np.float32))
    d_gamma = cuda.to_device(gamma.astype(np.float32))
    d_beta = cuda.to_device(beta.astype(np.float32))
    d_out = cuda.device_array_like(out.astype(np.float32))
    _layer_norm_welford_kernel[Bsz, threads](d_x, d_gamma, d_beta, eps, d_out)
    cuda.synchronize()
    d_out.copy_to_host(out)
    return out


# --- Task 48: Sparse Matrix-Vector Multiplication (SpMV) in CSR Format ---
@cuda.jit
def _spmv_csr_kernel(row_ptr, col_indices, values, x, y, num_rows):
    row = cuda.grid(1)
    if row < num_rows:
        acc = 0.0
        start = row_ptr[row]
        end = row_ptr[row + 1]
        for k in range(start, end):
            acc += values[k] * x[col_indices[k]]
        y[row] = acc


def spmv_csr_kernel(row_ptr, col_indices, values, x, y, num_rows):
    """Host wrapper: y = A @ x for A in CSR format, thread-per-row."""
    bpg = (num_rows + TPB - 1) // TPB
    d_rowptr = cuda.to_device(row_ptr.astype(np.int32))
    d_cols = cuda.to_device(col_indices.astype(np.int32))
    d_vals = cuda.to_device(values.astype(np.float32))
    d_x = cuda.to_device(x.astype(np.float32))
    d_y = cuda.device_array_like(y.astype(np.float32))
    _spmv_csr_kernel[bpg, TPB](d_rowptr, d_cols, d_vals, d_x, d_y, num_rows)
    cuda.synchronize()
    d_y.copy_to_host(y)
    return y


# --- Task 49: Scaled Dot-Product Attention with Shared Memory Tiling ---
ATTN_TILE = 32


@cuda.jit
def _attention_sdpa_kernel(Q, K, V, Out, scale):
    row = cuda.blockIdx.x  # one query row per block
    tid = cuda.threadIdx.x
    S, D = Q.shape
    block_size = cuda.blockDim.x

    s_k = cuda.shared.array((ATTN_TILE, 64), dtype=np.float32)
    s_v = cuda.shared.array((ATTN_TILE, 64), dtype=np.float32)
    s_score = cuda.shared.array((ATTN_TILE,), dtype=np.float32)

    q_local = cuda.local.array((64,), dtype=np.float32)
    for d in range(D):
        q_local[d] = Q[row, d]

    running_max = -3.4e38
    running_sum = 0.0
    acc = cuda.local.array((64,), dtype=np.float32)
    for d in range(D):
        acc[d] = 0.0

    n_tiles = (S + ATTN_TILE - 1) // ATTN_TILE
    for t in range(n_tiles):
        k_idx = t * ATTN_TILE + tid
        if tid < ATTN_TILE:
            if k_idx < S:
                for d in range(D):
                    s_k[tid, d] = K[k_idx, d]
                    s_v[tid, d] = V[k_idx, d]
            else:
                for d in range(D):
                    s_k[tid, d] = 0.0
                    s_v[tid, d] = 0.0
        cuda.syncthreads()

        if tid < ATTN_TILE:
            kk = t * ATTN_TILE + tid
            if kk < S:
                dot = 0.0
                for d in range(D):
                    dot += q_local[d] * s_k[tid, d]
                s_score[tid] = dot * scale
            else:
                s_score[tid] = -3.4e38
        cuda.syncthreads()

        if tid == 0:
            tile_max = -3.4e38
            for kk in range(ATTN_TILE):
                if s_score[kk] > tile_max:
                    tile_max = s_score[kk]

            new_max = tile_max if tile_max > running_max else running_max
            correction = math.exp(running_max - new_max)
            running_sum *= correction
            for d in range(D):
                acc[d] *= correction

            for kk in range(ATTN_TILE):
                k_global = t * ATTN_TILE + kk
                if k_global < S:
                    w = math.exp(s_score[kk] - new_max)
                    running_sum += w
                    for d in range(D):
                        acc[d] += w * s_v[kk, d]
            running_max = new_max
        cuda.syncthreads()

    if tid == 0:
        for d in range(D):
            Out[row, d] = acc[d] / running_sum


def attention_sdpa_tiled(Q, K, V, Out, scale):
    """Host wrapper: Out = softmax(Q K^T * scale) V, with online-softmax tiling."""
    S, D = Q.shape
    assert D <= 64, "demo kernel supports head_dim <= 64 (local-array size)"
    d_Q = cuda.to_device(Q.astype(np.float32))
    d_K = cuda.to_device(K.astype(np.float32))
    d_V = cuda.to_device(V.astype(np.float32))
    d_Out = cuda.device_array_like(Out.astype(np.float32))
    _attention_sdpa_kernel[S, ATTN_TILE](d_Q, d_K, d_V, d_Out, scale)
    cuda.synchronize()
    d_Out.copy_to_host(Out)
    return Out


# --- Task 50: Fused Rotary Position Embedding (RoPE) Kernel ---
@cuda.jit
def _rope_kernel(q, cos_table, sin_table, q_rot):
    s, d_half = cuda.grid(2)
    S, D = q.shape
    half = D // 2
    if s < S and d_half < half:
        x1 = q[s, 2 * d_half]
        x2 = q[s, 2 * d_half + 1]
        c = cos_table[s, d_half]
        sn = sin_table[s, d_half]
        q_rot[s, 2 * d_half] = x1 * c - x2 * sn
        q_rot[s, 2 * d_half + 1] = x1 * sn + x2 * c


def fused_rope_embedding(q, cos_table, sin_table, q_rot):
    """Host wrapper: applies 2D RoPE rotation to adjacent-pair elements of q."""
    S, D = q.shape
    half = D // 2
    threads = (16, 16)
    blocks = ((S + 15) // 16, (half + 15) // 16)
    d_q = cuda.to_device(q.astype(np.float32))
    d_cos = cuda.to_device(cos_table.astype(np.float32))
    d_sin = cuda.to_device(sin_table.astype(np.float32))
    d_qrot = cuda.device_array_like(q_rot.astype(np.float32))
    _rope_kernel[blocks, threads](d_q, d_cos, d_sin, d_qrot)
    cuda.synchronize()
    d_qrot.copy_to_host(q_rot)
    return q_rot


In [ ]:
# --- Module 6 Verification ---
import scipy.special
import scipy.sparse

print("Module 6: Deep Learning & Tensor Primitives — verification")

# Task 43
Bsz, V = 8, 1000
logits = (np.random.randn(Bsz, V) * 10).astype(np.float32)
probs = np.empty_like(logits)
stable_online_softmax(logits, probs)
np.testing.assert_allclose(probs, scipy.special.softmax(logits, axis=-1), atol=1e-5)
print("[PASS] Task 43: stable_online_softmax")

# Task 44
Bsz, D = 32, 128
x = np.random.randn(Bsz * D).astype(np.float32)
bias = np.random.randn(D).astype(np.float32)
out = np.empty_like(x)
fused_bias_gelu(x, bias, out)
y = x + np.tile(bias, Bsz)
ref = 0.5 * y * (1 + np.tanh(np.sqrt(2 / np.pi) * (y + 0.044715 * y**3)))
np.testing.assert_allclose(out, ref, atol=1e-4)
print("[PASS] Task 44: fused_bias_gelu")


def cpu_maxpool2d(x, ph, pw, stride):
    B, C, H, W = x.shape
    OH, OW = (H - ph) // stride + 1, (W - pw) // stride + 1
    out = np.empty((B, C, OH, OW), np.float32)
    idx = np.empty((B, C, OH, OW), np.int32)
    for oh in range(OH):
        for ow in range(OW):
            win = x[:, :, oh*stride:oh*stride+ph, ow*stride:ow*stride+pw].reshape(B, C, -1)
            a = win.argmax(-1)
            out[:, :, oh, ow] = win.max(-1)
            idx[:, :, oh, ow] = (oh*stride + a // pw) * W + (ow*stride + a % pw)
    return out, idx


# Task 45
B, C, H, W = 2, 3, 16, 16
x = np.random.rand(B, C, H, W).astype(np.float32)
out = np.empty((B, C, 8, 8), dtype=np.float32)
argmax_idx = np.empty((B, C, 8, 8), dtype=np.int32)
maxpool2d_with_argmax(x, out, argmax_idx, 2, 2, 2)
ref_out, ref_idx = cpu_maxpool2d(x, 2, 2, 2)
np.testing.assert_allclose(out, ref_out)
np.testing.assert_equal(argmax_idx, ref_idx)
print("[PASS] Task 45: maxpool2d_with_argmax")

# Task 46
B, D = 128, 64
x = np.random.randn(B, D).astype(np.float32)
gamma = np.random.rand(D).astype(np.float32)
beta = np.random.rand(D).astype(np.float32)
out = np.empty_like(x)
mean_out = np.empty(D, dtype=np.float32)
var_out = np.empty(D, dtype=np.float32)
batch_norm_1d_forward(x, gamma, beta, 1e-5, out, mean_out, var_out)
ref = gamma * (x - x.mean(0)) / np.sqrt(x.var(0) + 1e-5) + beta
np.testing.assert_allclose(out, ref, atol=1e-4)
print("[PASS] Task 46: batch_norm_1d_forward")

# Task 47
B, D = 16, 768
x = np.random.randn(B, D).astype(np.float32)
gamma = np.random.rand(D).astype(np.float32)
beta = np.random.rand(D).astype(np.float32)
out = np.empty_like(x)
layer_norm_welford(x, gamma, beta, 1e-5, out)
mu = x.mean(-1, keepdims=True)
var = x.var(-1, keepdims=True)
ref = gamma * (x - mu) / np.sqrt(var + 1e-5) + beta
np.testing.assert_allclose(out, ref, atol=1e-5)
print("[PASS] Task 47: layer_norm_welford")

# Task 48
A = scipy.sparse.random(500, 400, density=0.05, format="csr", dtype=np.float32)
xv = np.random.rand(400).astype(np.float32)
yv = np.empty(500, dtype=np.float32)
spmv_csr_kernel(A.indptr, A.indices, A.data, xv, yv, 500)
np.testing.assert_allclose(yv, A.dot(xv), atol=1e-4)
print("[PASS] Task 48: spmv_csr_kernel")

# Task 49
S, D = 100, 64
Q = np.random.randn(S, D).astype(np.float32)
K = np.random.randn(S, D).astype(np.float32)
Vv = np.random.randn(S, D).astype(np.float32)
Out = np.empty_like(Q)
scale = 1.0 / np.sqrt(D)
attention_sdpa_tiled(Q, K, Vv, Out, scale)
ref = scipy.special.softmax(Q @ K.T * scale, axis=-1) @ Vv
np.testing.assert_allclose(Out, ref, atol=1e-3)
print("[PASS] Task 49: attention_sdpa_tiled")

# Task 50
S, D = 64, 32
q = np.random.randn(S, D).astype(np.float32)
pos = np.arange(S)[:, None]
inv_freq = 1.0 / (10000 ** (np.arange(D // 2) / (D // 2)))
cos_table = np.cos(pos * inv_freq).astype(np.float32)
sin_table = np.sin(pos * inv_freq).astype(np.float32)
q_rot = np.empty_like(q)
fused_rope_embedding(q, cos_table, sin_table, q_rot)
ref = np.empty_like(q)
ref[:, 0::2] = q[:, 0::2] * cos_table - q[:, 1::2] * sin_table
ref[:, 1::2] = q[:, 0::2] * sin_table + q[:, 1::2] * cos_table
np.testing.assert_allclose(q_rot, ref, atol=1e-5)
print("[PASS] Task 50: fused_rope_embedding")

print("\nModule 6: all 8 tasks passed.")
